In [1]:
from datetime import datetime
import MetaTrader5 as mt5
import pandas as pd
import pytz
import time
import pandas_ta as pta
import matplotlib.pyplot as plt 
# import threading
mt5.initialize()


True

In [2]:
def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

In [3]:
def price_action(symbol, lot, ask, bid, order_type):
    buy_profit=mt5.order_calc_profit(order_type,symbol,lot,ask,bid)
    return buy_profit
price_action("GBPJPY", 1.0, 1.18969, 1.17030,mt5.ORDER_TYPE_SELL)

13.29

In [4]:
def calculate_heikin_ashi(df):
    ha_close = (df['open'] + df['high'] + df['low'] + df['close']) / 4
    ha_open = (ha_close.shift(1) + ha_close.shift(1)) / 2
    ha_high = df[['high', 'open', 'close']].max(axis=1)
    ha_low = df[['low', 'open', 'close']].min(axis=1)

    return pd.DataFrame({'ha_open': ha_open, 'ha_high': ha_high, 'ha_low': ha_low, 'ha_close': ha_close, 'time':df.time})

In [ ]:
import numpy as np
import pandas as pd
import pandas_ta as pdt

def supertrend(factor, atr_length, high, low, close):
    atr = pdt.atr(high, low, close, atr_length)
    basic_upper_band = (high + low) / 2 + factor * atr
    basic_lower_band = (high + low) / 2 - factor * atr
    bullish_signal = close > basic_upper_band
    bearish_signal = close < basic_lower_band
    bullish_supertrend = np.full_like(close, np.nan)
    bearish_supertrend = np.full_like(close, np.nan)

    for i in range(1, len(close)):
        if bullish_signal[i] or (bullish_supertrend[i-1] and close[i-1] > basic_upper_band[i-1]):
            bullish_supertrend[i] = max(basic_upper_band[i], bullish_supertrend[i-1])
        else:
            bullish_supertrend[i] = basic_upper_band[i]

        if bearish_signal[i] or (bearish_supertrend[i-1] and close[i-1] < basic_lower_band[i-1]):
            bearish_supertrend[i] = min(basic_lower_band[i], bearish_supertrend[i-1])
        else:
            bearish_supertrend[i] = basic_lower_band[i]

    direction = np.where(close > bullish_supertrend, 1, np.where(close < bearish_supertrend, -1, 0))
    supertrend = np.where(direction == 1, bullish_supertrend, bearish_supertrend)
    
    return supertrend, direction

# Example usage:
# Assuming df is your DataFrame containing OHLC data
# Replace this with your actual DataFrame
# Example:
# df = pd.DataFrame({'open': [...], 'high': [...], 'low': [...], 'close': [...]})

# Convert input parameters from Pine Script to Python
# factor = 3.0
# atr_length = 10




In [ ]:
pdt.atr?

In [ ]:
def get_values(symbol, size, smaa=150, t='M5'):
    d = {'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
#     rates_frame['ema'] =ema(rates_frame['close'], 200)
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
#     rates_frame = rates_frame[rates_frame['sma'].notna()]
    # Calculate Supertrend
    factor = 3.0
    atr_length = 10
    supertrend_values, direction = supertrend(factor, atr_length, rates_frame['high'], rates_frame['low'], rates_frame['close'])
    rates_frame['spvalues'] = supertrend_values
    rates_frame['direction'] = direction
    
    # Print or access the supertrend_values and direction arrays
    print("Supertrend values:", supertrend_values)
    print("Direction:", direction)
    return rates_frame

In [71]:
def get_values(symbol, size, smaa=50, t='M30'):
    d = {'M1':mt5.TIMEFRAME_M1, 'M5':mt5.TIMEFRAME_M5,'M30':mt5.TIMEFRAME_M30, 'M15':mt5.TIMEFRAME_M15,'H1':mt5.TIMEFRAME_H1, 'H2':mt5.TIMEFRAME_H2, 'H3':mt5.TIMEFRAME_H3, 'H4':mt5.TIMEFRAME_H4, 'H12':mt5.TIMEFRAME_H12, 'D1':mt5.TIMEFRAME_D1, 'W1':mt5.TIMEFRAME_W1}
    rates = mt5.copy_rates_from_pos(symbol, d[t], 0, size)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    
    rates_frame['ema'] =rates_frame['close'].ewm(span=15, adjust=False).mean()
#     rates_frame['ema'] =rates_frame['close'].ewm(span=200, adjust=False).mean()
    
#     rates_frame['ema'] =ema(rates_frame['close'], 15)
#     rates_frame['ema'] =ema(rates_frame['close'], 9)
    rates_frame['rsi1'] = get_rsi(rates_frame['close'], 21)
    rates_frame['rsi2'] = get_rsi(rates_frame['close'], 14)
    
    
    rates_frame['sma'] = rates_frame['close'].rolling(window=smaa).mean()
    rates_frame = rates_frame[rates_frame['sma'].notna()]
#         print(rates_frame.head())
    # Calculate Supertren
    return rates_frame

In [6]:
def ema(s, n):
    ema = []
    zero = [0]*(20000-19801)
    j = 1

    #get n sma first and calculate the next n period ema
    sma = sum(s[:n]) / n
    multiplier = 2 / float(1 + n)
    ema.append(sma)

    #EMA(current) = ( (Price(current) - EMA(prev) ) x Multiplier) + EMA(prev)
    ema.append(( (s[n] - sma) * multiplier) + sma)

    #now calculate the rest of the values
    for i in s[n+1:]:
        tmp = ( (i - ema[j]) * multiplier) + ema[j]
        j = j + 1
        ema.append(tmp)
    am = zero + ema
    print(len(am))
    return am

In [7]:
def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

In [ ]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 2000, 25, 'M1')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0
# 
for i in range(1, len(a)):
    if check==0:
        if a.iloc[i-1].close <= a.iloc[i-1].sma and direction(a,i-1)==0:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].open
            check=1
            
        if a.iloc[i-1].close >= a.iloc[i-1].sma and direction(a,i-1)==1:
            print(f"{a.iloc[i].name}")
            buy_price = a.iloc[i].open
            check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

    if check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-10:
            profit.append(-10)
        else:
            profit.append(pp)
        check=0

In [22]:
NEED TO REVISIT THIS AGAIN, RSI1 AND 14 LESS THAN 50 AT THE SAME TIME
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 20000, 25, 'M1')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)


for i in range(1, len(a)):
    if check==0:
        if a.iloc[i-3].rsi1 >50.0 and a.iloc[i-3].rsi2 > 50.0 and a.iloc[i-2].rsi1 < 50.0 and a.iloc[i-1].rsi1 < a.iloc[i-2].rsi1 and a.iloc[i-1].rsi2 < 50 and a.iloc[i-1].rsi2 < a.iloc[i-2].rsi2:
            print("=="*20)
            print(f"{a.iloc[i].name}")
            c = 0
           
            buy_price = a.iloc[i].close
            check=1
            
#         if a.iloc[i-1].close >= a.iloc[i-1].sma and direction(a,i-1)==1:
#             print(f"{a.iloc[i].name}")
#             buy_price = a.iloc[i].open
#             check=2
#             continue
    if check==1:
        sell_price = a.iloc[i].close
        lot = 0.1
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10))
#         ppb = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
        print(f"PP {pp}--{ a.iloc[i].rsi2}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp > 0.50:
            c = 1
        if pp <= -3.0:
            print(f"{pp}--{ a.iloc[i].rsi2}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            if c !=1:
                profit.append(-3)
            else:
                profit.append(0.30)
            check = 0
        if (a.iloc[i].ema + 24.0) < sell_price:
#             ppp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10))
#             print(f"PPP {ppp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            if pp<0.0:
                if c!=1:
                    profit.append(pp)
                else:
                    profit.append(0.30)
            else:
                profit.append(pp)
            check =0
#         elif pp<-15:
#             profit.append(-15)
#             check =0
#         else:
#             profit.append(pp)
#         check=0

    if check==2:
        sell_price = a.iloc[i].close
        pp = price_action(symbol, 0.1, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - 2.50
        print(f"{pp}--{ a.iloc[i].sma}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
        if pp >= 0.0:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
            profit.append(pp)
            check = 0
#         elif a.iloc[i].close > a.iloc[i].ema:
#             print(f"{pp}--{ a.iloc[i].ema}---{a.iloc[i].close}--{buy_price}--{a.iloc[i].name}")
#             profit.append(pp)
        if pp<-5:
            profit.append(-5)
        else:
            profit.append(pp)
        check=0

                         open      high       low     close
time                                                       
2024-08-20 15:38:00  60676.80  60678.38  60666.59  60669.03
2024-08-20 15:39:00  60669.06  60674.09  60655.34  60673.66
2024-08-20 15:40:00  60675.06  60687.69  60662.83  60683.61
2024-08-20 15:41:00  60683.49  60726.29  60683.43  60700.98
2024-08-20 15:42:00  60700.54  60761.15  60697.90  60756.65
2024-08-20 17:02:00
PP -2.2--45.58364612776246---60690.78--60690.78--2024-08-20 17:02:00
PP -12.649999999999999--52.100259134930546---60795.27--60690.78--2024-08-20 17:03:00
-12.649999999999999--52.100259134930546---60795.27--60690.78--2024-08-20 17:03:00
2024-08-20 19:40:00
PP -2.2--41.92509874763455---58818.07--58818.07--2024-08-20 19:40:00
PP -2.6500000000000004--42.397304426820746---58822.54--58818.07--2024-08-20 19:41:00
PP -1.4800000000000002--41.44973188405006---58810.87--58818.07--2024-08-20 19:42:00
PP -3.3000000000000003--43.57250535742466---58829.11--58818.07--20

2024-08-21 05:56:00
PP -2.2--43.417149099666965---59361.12--59361.12--2024-08-21 05:56:00
PP -4.7--48.396614625141595---59386.12--59361.12--2024-08-21 05:57:00
-4.7--48.396614625141595---59386.12--59361.12--2024-08-21 05:57:00
2024-08-21 06:20:00
PP -2.2--37.480480516884796---59313.2--59313.2--2024-08-21 06:20:00
PP -5.09--44.03585134034078---59342.07--59313.2--2024-08-21 06:21:00
-5.09--44.03585134034078---59342.07--59313.2--2024-08-21 06:21:00
2024-08-21 07:07:00
PP -2.2--46.2614515820756---59259.88--59259.88--2024-08-21 07:07:00
PP -2.0100000000000002--45.80002149535073---59257.95--59259.88--2024-08-21 07:08:00
PP -0.14000000000000012--41.48680831504257---59239.27--59259.88--2024-08-21 07:09:00
PP 1.4--38.281863898622355---59223.85--59259.88--2024-08-21 07:10:00
PP 2.8499999999999996--35.513210625965556---59209.4--59259.88--2024-08-21 07:11:00
PP 2.5699999999999994--36.46523605565835---59212.18--59259.88--2024-08-21 07:12:00
PP 2.6100000000000003--36.373707222753445---59211.74--5925

2024-08-21 18:29:00
PP -2.2--53.83456238130643---59534.9--59534.9--2024-08-21 18:29:00
2024-08-21 18:50:00
PP -2.2--44.24553170970921---59585.34--59585.34--2024-08-21 18:50:00
PP -1.7000000000000002--43.786984191353376---59580.38--59585.34--2024-08-21 18:51:00
PP -1.8400000000000003--43.96477045535565---59581.79--59585.34--2024-08-21 18:52:00
PP -6.44--49.56422365367668---59627.75--59585.34--2024-08-21 18:53:00
-6.44--49.56422365367668---59627.75--59585.34--2024-08-21 18:53:00
2024-08-21 20:04:00
PP -2.2--46.37349222080183---59757.85--59757.85--2024-08-21 20:04:00
PP -1.27--45.49936665750506---59748.54--59757.85--2024-08-21 20:05:00
PP -0.8900000000000001--45.131331951999265---59744.8--59757.85--2024-08-21 20:06:00
PP 0.9299999999999997--43.288257822637576---59726.52--59757.85--2024-08-21 20:07:00
PP 1.46--42.74831048686214---59721.27--59757.85--2024-08-21 20:08:00
PP -1.1700000000000002--46.35283647680241---59747.53--59757.85--2024-08-21 20:09:00
PP -3.4800000000000004--49.37489967501

PP 2.2--51.13130467978127---60689.62--60733.64--2024-08-22 08:56:00
PP 1.5099999999999998--52.84074271941443---60696.5--60733.64--2024-08-22 08:57:00
PP -1.1500000000000001--58.85382210819598---60723.19--60733.64--2024-08-22 08:58:00
2024-08-22 09:57:00
PP -2.2--46.631366934095446---60768.8--60768.8--2024-08-22 09:57:00
PP -2.7--48.02275426315148---60773.75--60768.8--2024-08-22 09:58:00
PP -3.1--49.20122286205948---60777.84--60768.8--2024-08-22 09:59:00
-3.1--49.20122286205948---60777.84--60768.8--2024-08-22 09:59:00
2024-08-22 10:09:00
PP -2.2--39.05336682415553---60744.8--60744.8--2024-08-22 10:09:00
PP -1.1400000000000001--37.17262211626364---60734.18--60744.8--2024-08-22 10:10:00
PP -0.010000000000000231--35.236736319969694---60722.93--60744.8--2024-08-22 10:11:00
PP -1.8900000000000001--40.78371662191836---60741.72--60744.8--2024-08-22 10:12:00
PP -2.4800000000000004--42.43676979380841---60747.57--60744.8--2024-08-22 10:13:00
PP -3.92--46.402996176336956---60761.97--60744.8--2024-

2024-08-22 22:55:00
PP -2.2--42.476806490044446---60254.6--60254.6--2024-08-22 22:55:00
PP -5.470000000000001--47.33905917759461---60287.3--60254.6--2024-08-22 22:56:00
-5.470000000000001--47.33905917759461---60287.3--60254.6--2024-08-22 22:56:00
2024-08-22 23:01:00
PP -2.2--54.87816589640786---60341.03--60341.03--2024-08-22 23:01:00
2024-08-22 23:39:00
PP -2.2--49.340661758771425---60360.03--60360.03--2024-08-22 23:39:00
PP -2.1500000000000004--49.249311621278856---60359.51--60360.03--2024-08-22 23:40:00
PP -1.0700000000000003--47.29968137165651---60348.76--60360.03--2024-08-22 23:41:00
PP -0.6700000000000002--46.56107793081801---60344.76--60360.03--2024-08-22 23:42:00
PP -0.7700000000000002--46.771431769354194---60345.7--60360.03--2024-08-22 23:43:00
PP -2.3000000000000003--50.21758009077869---60361.05--60360.03--2024-08-22 23:44:00
PP -5.27--56.122799595626425---60390.68--60360.03--2024-08-22 23:45:00
-5.27--56.122799595626425---60390.68--60360.03--2024-08-22 23:45:00
2024-08-23 00:

...

PP 12.16--30.938074741729125---57332.68--57476.24--2024-09-02 05:34:00
PP 10.55--33.563277734199076---57348.77--57476.24--2024-09-02 05:35:00
PP 12.239999999999998--32.17650552060485---57331.83--57476.24--2024-09-02 05:36:00
PP 13.219999999999999--31.36660246130313---57322.0--57476.24--2024-09-02 05:37:00
PP 17.01--28.400035338421134---57284.12--57476.24--2024-09-02 05:38:00
PP 16.73--28.944542645013584---57286.97--57476.24--2024-09-02 05:39:00
PP 14.27--33.639467425100406---57311.59--57476.24--2024-09-02 05:40:00
PP 14.350000000000001--33.5618670297965---57310.79--57476.24--2024-09-02 05:41:00
PP 17.150000000000002--30.8735863612236---57282.75--57476.24--2024-09-02 05:42:00
PP 14.530000000000001--36.02050904348997---57308.9--57476.24--2024-09-02 05:43:00
PP 11.2--41.95939202813233---57342.27--57476.24--2024-09-02 05:44:00
PP 12.16--40.7805045024965---57332.62--57476.24--2024-09-02 05:45:00
PP 7.079999999999999--48.91567125353472---57383.41--57476.24--2024-09-02 05:46:00
2024-09-02 06:

PP 1.71--50.064079089767176---58584.62--58623.76--2024-09-02 18:05:00
PP -2.04--53.44411951225184---58622.2--58623.76--2024-09-02 18:06:00
2024-09-02 18:16:00
PP -2.2--44.65762807133662---58561.45--58561.45--2024-09-02 18:16:00
PP -6.03--49.08119512945472---58599.71--58561.45--2024-09-02 18:17:00
-6.03--49.08119512945472---58599.71--58561.45--2024-09-02 18:17:00
2024-09-02 18:28:00
PP -2.2--43.61824666808733---58553.81--58553.81--2024-09-02 18:28:00
PP 0.41999999999999993--41.025948813414054---58527.6--58553.81--2024-09-02 18:29:00
PP -3.43--46.0963125164743---58566.12--58553.81--2024-09-02 18:30:00
-3.43--46.0963125164743---58566.12--58553.81--2024-09-02 18:30:00
2024-09-02 18:48:00
PP -2.2--47.509606587259576---58600.46--58600.46--2024-09-02 18:48:00
PP -3.8200000000000003--49.71254498355142---58616.69--58600.46--2024-09-02 18:49:00
-3.8200000000000003--49.71254498355142---58616.69--58600.46--2024-09-02 18:49:00
2024-09-02 18:54:00
PP -2.2--49.330151259952785---58611.12--58611.12--20

PP 11.719999999999999--37.1069709589462---59424.13--59563.3--2024-09-03 05:48:00
PP 9.68--41.12537167721838---59444.47--59563.3--2024-09-03 05:49:00
PP 9.36--41.75587383641603---59447.67--59563.3--2024-09-03 05:50:00
PP 5.86--48.28327442683294---59482.69--59563.3--2024-09-03 05:51:00
PP 8.510000000000002--44.24967781609888---59456.24--59563.3--2024-09-03 05:52:00
PP 15.2--36.0450894734776---59389.32--59563.3--2024-09-03 05:53:00
PP 16.19--35.00883883218883---59379.4--59563.3--2024-09-03 05:54:00
PP 16.970000000000002--34.172683689872414---59371.56--59563.3--2024-09-03 05:55:00
PP 18.94--32.099123397503135---59351.87--59563.3--2024-09-03 05:56:00
PP 23.03--28.268583614813622---59311.04--59563.3--2024-09-03 05:57:00
PP 25.42--26.28620411512746---59287.08--59563.3--2024-09-03 05:58:00
PP 24.7--27.924172520589693---59294.29--59563.3--2024-09-03 05:59:00
PP 24.720000000000002--27.906574399835776---59294.1--59563.3--2024-09-03 06:00:00
PP 27.2--25.636441261998016---59269.31--59563.3--2024-09

PP 24.85--41.76068953512597---58778.91--59049.43--2024-09-03 11:58:00
PP 27.25--38.8800745780612---58754.91--59049.43--2024-09-03 11:59:00
PP 24.7--43.35937253755695---58780.46--59049.43--2024-09-03 12:00:00
PP 22.62--46.774280129306334---58801.23--59049.43--2024-09-03 12:01:00
PP 21.44--48.67554145380521---58813.08--59049.43--2024-09-03 12:02:00
PP 19.990000000000002--50.98283471232413---58827.58--59049.43--2024-09-03 12:03:00
2024-09-03 12:58:00
PP -2.2--44.20876951354325---58871.93--58871.93--2024-09-03 12:58:00
PP -1.5700000000000003--42.95164804222081---58865.64--58871.93--2024-09-03 12:59:00
PP 0.06999999999999984--39.76858794067242---58849.2--58871.93--2024-09-03 13:00:00
PP 1.79--36.7071744469968---58832.02--58871.93--2024-09-03 13:01:00
PP 0.6199999999999997--40.09509590875345---58843.74--58871.93--2024-09-03 13:02:00
PP -4.78--52.66315263776418---58897.72--58871.93--2024-09-03 13:03:00
-4.78--52.66315263776418---58897.72--58871.93--2024-09-03 13:03:00
2024-09-03 14:37:00
PP -

2024-09-03 18:20:00
PP -2.2--43.53380462239721---57744.4--57744.4--2024-09-03 18:20:00
PP -8.11--49.23727994931845---57803.52--57744.4--2024-09-03 18:21:00
-8.11--49.23727994931845---57803.52--57744.4--2024-09-03 18:21:00
2024-09-03 18:41:00
PP -2.2--41.211290896366656---57694.6--57694.6--2024-09-03 18:41:00
PP -1.2600000000000002--40.510894120327265---57685.16--57694.6--2024-09-03 18:42:00
PP 3.6399999999999997--37.0007396377167---57636.23--57694.6--2024-09-03 18:43:00
PP -0.2100000000000002--41.30257055178808---57674.66--57694.6--2024-09-03 18:44:00
PP -9.56--50.216239711051536---57768.23--57694.6--2024-09-03 18:45:00
-9.56--50.216239711051536---57768.23--57694.6--2024-09-03 18:45:00


In [380]:
# Minute 1
#less negatives more positives
# symbol = "CADJPY"
# b= get_values1(symbol)
# b = b.dropna()
import threading
# b = a
B = []
check = 0
global  profit
profit = []
global index
index = []
indexB = []
counter = 0
peck = 0
global p
p= []
checks = 0
counterr = 0
profits = []
up = 0
rsi1 = []
rsi2 = []
mul = 100000
global_loss = []
conti = []
global max_loss
max_loss = []
pp_old = 0.0
symbol = "BTCUSD"
a = get_values(symbol, 50000, 25, 'M15')
timezone =pytz.timezone('Etc/GMT-2')
# create 'datetime' objects in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2023, 10, 30, hour=6, minute=30, tzinfo=timezone)


for i in range(10, len(a)-5):
    if check==0:
#         print(str(a.iloc[i+1].name.time())[:6])
        if "00:00" in str(a.iloc[i+1].name.time())[:6] and a.iloc[i].close < a.iloc[i].open:
            print("=="*20)
            print(f"{a.iloc[i].name} !! {a.iloc[i].rsi1} !! {a.iloc[i].rsi2} !! {a.iloc[i].close}")
            c = 0
            buy_price = a.iloc[i].close
            pp_old = 0.0
            check=1
            checks = 0
            up = 0
            
#         if "00:05" in str(a.iloc[i+1].name.time())[:6] and a.iloc[i].close > a.iloc[i].open:
#             print("=="*20)
#             print(f"{a.iloc[i].name} !! {a.iloc[i].rsi1} !! {a.iloc[i].rsi2}")
#             c = 0
#             buy_price = a.iloc[i].close
#             check=2
#             continue
    elif check==1:
        sell_price = a.iloc[i].close
        lot = 1
        pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_SELL) - ((2.20)*(lot*10))
        ppopen = price_action(symbol, 2, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
        print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price}--{a.iloc[i].name}")
        if pp> 0.0:
            checks=1
        if a.iloc[i].rsi1>a.iloc[i-1].rsi1:
            ppbet = price_action(symbol, 2, a.iloc[i].close, a.iloc[i+1].close, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
            print(f"PPBET {ppbet}-- { a.iloc[i+1].rsi1}--- {a.iloc[i+2].rsi1}--{ a.iloc[i+3].rsi1} !! {sell_price}--{a.iloc[i].name}")
            profit.append(ppbet)
            profits.append(ppbet)
            if ppbet <0.0:
                up =1
        if pp < pp_old/2 and checks!=0:
            print(f"PP_old {pp_old/2}-- { a.iloc[i+1].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
            profit.append(pp_old/2)
            check =0
            continue
        if c==0 and pp <-100:
#             if ppopen<0.0:
            profit.append(ppopen)
            profit.append(pp)
            print(f"PPOPEN {ppopen}-- { a.iloc[i+1].rsi1}--- {a.iloc[i+1].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i+1].name}")
            check =0
            continue
        
        if c==3 or pp < -5*(lot*10) or pp >=100*(lot*10):

            print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price} !! {sell_price}--{a.iloc[i].name}")
            profit.append(pp)
            check =0

        pp_old = pp
        c+=1

#     elif check==2:
#         sell_price = a.iloc[i].close
#         lot = 1
#         pp = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
# #         ppb = price_action(symbol, lot, buy_price, sell_price, mt5.ORDER_TYPE_BUY) - ((2.20)*(lot*10))
#         print(f"PP {pp}-- { a.iloc[i].rsi1}--- {a.iloc[i].rsi2}--{buy_price}--{a.iloc[i].name}")
        
#         if c==6 or pp < -5*(lot*10) or pp >=100*(lot*10):
#             profit.append(pp)
#             check =0
#         c+=1

2023-03-28 23:45:00 !! 60.873182317730155 !! 62.96812678645674 !! 27303.57
PP 39.32-- 57.56813406751497--- 58.055202632731564--27303.57--2023-03-29 00:00:00
PP 28.92-- 57.97447024119062--- 58.64454540443193--27303.57--2023-03-29 00:15:00
PPBET -24.78-- 57.89666342356125--- 54.407852171824096--54.59408190693022 !! 27252.65--2023-03-29 00:15:00
PP 30.310000000000002-- 57.89666342356125--- 58.526185059775614--27303.57--2023-03-29 00:30:00
PP 93.56-- 54.407852171824096--- 53.25874129049448--27303.57--2023-03-29 00:45:00
PP 93.56-- 54.407852171824096--- 53.25874129049448--27303.57 !! 27188.01--2023-03-29 00:45:00
2023-04-01 23:45:00 !! 53.80642591594601 !! 56.2645273688938 !! 28442.49
PP -62.51-- 57.380988159863826--- 61.18819213353391--28442.49--2023-04-02 00:00:00
PPBET -11.88-- 57.80917848729397--- 57.32870388597345--57.39344964048837 !! 28483.0--2023-04-02 00:00:00
PP -62.51-- 57.380988159863826--- 61.18819213353391--28442.49 !! 28483.0--2023-04-02 00:00:00
2023-04-02 23:45:00 !! 41.480

PP -38.75-- 48.32113161617025--- 49.12785299590934--27491.78 !! 27508.53--2023-04-24 00:45:00
2023-04-27 23:45:00 !! 58.06269486741603 !! 58.10840105714777 !! 29595.34
PP -177.64-- 61.32535138194903--- 62.849600743474866--29595.34--2023-04-28 00:00:00
PPBET -124.82-- 59.714099623814235--- 57.03368824822742--57.659388856954095 !! 29750.98--2023-04-28 00:00:00
PPOPEN -124.82-- 59.714099623814235--- 60.417211619740804--29595.34 !! 29750.98--2023-04-28 00:15:00
2023-04-29 23:45:00 !! 45.28326233788101 !! 45.134970660515975 !! 29224.94
PP -37.89-- 47.11855533303359--- 47.87164624968706--29224.94--2023-04-30 00:00:00
PPBET -24.0-- 47.01435212995965--- 48.177993361237874--46.036186585104716 !! 29240.83--2023-04-30 00:00:00
PP -36.89-- 47.01435212995965--- 47.710358579196054--29224.94--2023-04-30 00:15:00
PP -46.56-- 48.177993361237874--- 49.48280530154959--29224.94--2023-04-30 00:30:00
PPBET -61.02-- 46.036186585104716--- 42.79455929899093--44.208210149006646 !! 29249.5--2023-04-30 00:30:00
P

2023-06-25 23:45:00 !! 39.08057410529594 !! 37.30435185413216 !! 30374.02
PP -82.02000000000001-- 43.47824523876761--- 43.81393380445989--30374.02--2023-06-26 00:00:00
PPBET 64.0-- 46.38948954981216--- 44.085123537953805--45.29751640128957 !! 30434.04--2023-06-26 00:00:00
PP -82.02000000000001-- 43.47824523876761--- 43.81393380445989--30374.02 !! 30434.04--2023-06-26 00:00:00
2023-06-26 23:45:00 !! 45.40403813390451 !! 44.92649907045616 !! 30147.64
PP -82.5-- 48.435242976801696--- 49.62277744410342--30147.64--2023-06-27 00:00:00
PPBET -44.260000000000005-- 47.92130291717227--- 45.72079297208181--45.517215319778835 !! 30208.14--2023-06-27 00:00:00
PP -82.5-- 48.435242976801696--- 49.62277744410342--30147.64 !! 30208.14--2023-06-27 00:00:00
2023-07-01 23:45:00 !! 54.052226244837634 !! 52.066905300963604 !! 30581.21
PP -44.25-- 57.10544089755416--- 57.21099412841035--30581.21--2023-07-02 00:00:00
PPBET -55.48-- 54.25727314696819--- 52.3714882204605--54.319541844639666 !! 30603.46--2023-07

2023-08-12 23:45:00 !! 46.171239321430576 !! 43.087998927538926 !! 29394.14
PP -27.45-- 47.86621056048706--- 45.89628547098125--29394.14--2023-08-13 00:00:00
PPBET -20.0-- 48.18057494635292--- 44.712648860740444--48.89063885288518 !! 29399.59--2023-08-13 00:00:00
PP -28.45-- 48.18057494635292--- 46.41872828628059--29394.14--2023-08-13 00:15:00
PPBET -46.5-- 44.712648860740444--- 48.89063885288518--50.376480743227695 !! 29400.59--2023-08-13 00:15:00
PP -16.2-- 44.712648860740444--- 41.17365167159163--29394.14--2023-08-13 00:30:00
PP -29.45-- 48.89063885288518--- 48.01582716441186--29394.14--2023-08-13 00:45:00
PPBET -12.0-- 50.376480743227695--- 50.33653062471741--49.042321579655685 !! 29401.59--2023-08-13 00:45:00
PP -29.45-- 48.89063885288518--- 48.01582716441186--29394.14 !! 29401.59--2023-08-13 00:45:00
2023-08-15 23:45:00 !! 38.20049351703561 !! 36.91734744840409 !! 29161.26
PP -27.13-- 38.90508802995889--- 37.979768750063194--29161.26--2023-08-16 00:00:00
PPBET 41.36-- 43.11081493

2023-10-08 23:45:00 !! 51.754507883881956 !! 53.319735064516294 !! 27910.97
PP -59.9-- 55.900561266390945--- 59.54148245701979--27910.97--2023-10-09 00:00:00
PPBET -69.16-- 52.92912283874185--- 58.0347489710581--51.79598644323512 !! 27948.87--2023-10-09 00:00:00
PP -59.9-- 55.900561266390945--- 59.54148245701979--27910.97 !! 27948.87--2023-10-09 00:00:00
2023-10-09 23:45:00 !! 50.11318080575599 !! 51.65719507710755 !! 27571.21
PP -18.88-- 49.88370277671592--- 51.289533600936934--27571.21--2023-10-10 00:00:00
PP -51.83-- 52.305551950572365--- 54.937245647521294--27571.21--2023-10-10 00:15:00
PPBET 55.64-- 54.995906668320046--- 53.74299429225144--52.04501219482346 !! 27601.04--2023-10-10 00:15:00
PP -51.83-- 52.305551950572365--- 54.937245647521294--27571.21 !! 27601.04--2023-10-10 00:15:00
2023-10-14 23:45:00 !! 43.22323590344141 !! 40.106405686378096 !! 26838.22
PP -44.29-- 48.15717406305667--- 47.913404201419254--26838.22--2023-10-15 00:00:00
PPBET -138.94-- 38.85676535489438--- 41.81

2023-11-21 23:45:00 !! 45.152925316108536 !! 44.41817253509089 !! 36842.83
PP 15.259999999999998-- 44.20800374564245--- 43.03633907900458--36842.83--2023-11-22 00:00:00
PP 215.78-- 39.53307869696291--- 36.46221889484456--36842.83--2023-11-22 00:15:00
PP 368.99-- 36.441444861623864--- 32.390864079881766--36842.83--2023-11-22 00:30:00
PP 438.63-- 35.1302431155579--- 30.712202486693556--36842.83--2023-11-22 00:45:00
PP 438.63-- 35.1302431155579--- 30.712202486693556--36842.83 !! 36382.2--2023-11-22 00:45:00
2023-11-23 23:45:00 !! 49.38099881332353 !! 49.386302322910744 !! 37252.36
PP -17.23-- 49.112316003583175--- 48.94212183594847--37252.36--2023-11-24 00:00:00
PP -5.780000000000001-- 48.44791314419869--- 47.83006814502881--37252.36--2023-11-24 00:15:00
PP -8.63-- 48.62954130870663--- 48.14589579937109--37252.36--2023-11-24 00:30:00
PPBET 113.80000000000001-- 52.7903868892018--- 57.06659993607129--52.658548083690924 !! 37238.99--2023-11-24 00:30:00
PP -76.53-- 52.7903868892018--- 55.1172

2024-01-29 23:45:00 !! 62.86623996525174 !! 63.19135413349914 !! 43174.96
PP 6.100000000000001-- 61.732348378644836--- 61.3797770870328--43174.96--2024-01-30 00:00:00
PP -8.32-- 62.10067554790596--- 61.982102062522124--43174.96--2024-01-30 00:15:00
PPBET -25.92-- 62.015486804746274--- 59.85657197775106--58.991494094687425 !! 43161.28--2024-01-30 00:15:00
PP_old 3.0500000000000007-- 62.015486804746274--- 61.982102062522124--43174.96 !! 43161.28--2024-01-30 00:15:00
2024-01-30 23:45:00 !! 52.78796149459354 !! 52.50217117178532 !! 43526.17
PP -14.18-- 52.48261552186822--- 52.04425550917645--43526.17--2024-01-31 00:00:00
PP 37.53-- 50.456198093595916--- 49.000820347760374--43526.17--2024-01-31 00:15:00
PP 38.56-- 50.41548544273355--- 48.93943036184247--43526.17--2024-01-31 00:30:00
PP 186.83-- 44.93515816850765--- 40.98023179425339--43526.17--2024-01-31 00:45:00
PP 186.83-- 44.93515816850765--- 40.98023179425339--43526.17 !! 43317.34--2024-01-31 00:45:00
2024-02-04 23:45:00 !! 45.539757133

2024-04-13 23:45:00 !! 19.395721097119093 !! 16.444756963383426 !! 61918.24
PP -245.13-- 21.82507064044607--- 19.520195651558993--61918.24--2024-04-14 00:00:00
PPBET -604.5-- 20.95929407002238--- 26.656415192336993--32.24193169438337 !! 62141.37--2024-04-14 00:00:00
PPOPEN -604.5-- 20.95929407002238--- 18.55990696264206--61918.24 !! 62141.37--2024-04-14 00:15:00
2024-04-14 23:45:00 !! 45.781469369757325 !! 44.10610127970486 !! 63856.67
PP 116.9-- 44.28303249550976--- 41.88811313768138--63856.67--2024-04-15 00:00:00
PP 185.24-- 43.54671328024598--- 40.800965803865054--63856.67--2024-04-15 00:15:00
PP 360.73-- 41.67816605031812--- 38.0686718834292--63856.67--2024-04-15 00:30:00
PP 145.34-- 44.73426298466152--- 43.10473263636386--63856.67--2024-04-15 00:45:00
PPBET 2509.62-- 58.237895579743686--- 61.334891304559505--59.190160305934334 !! 63689.33--2024-04-15 00:45:00
PP_old 180.365-- 58.237895579743686--- 43.10473263636386--63856.67 !! 63689.33--2024-04-15 00:45:00
2024-04-15 23:45:00 !! 

2024-07-08 23:45:00 !! 48.56563534721181 !! 48.29608161653848 !! 56236.78
PP -116.61-- 49.89266482643917--- 50.50917941573365--56236.78--2024-07-09 00:00:00
PPBET 618.18-- 54.099619561528215--- 50.239488879992955--53.547665682420295 !! 56331.39--2024-07-09 00:00:00
PPOPEN 618.18-- 54.099619561528215--- 57.186172829903605--56236.78 !! 56331.39--2024-07-09 00:15:00
2024-07-09 23:45:00 !! 55.97004660920116 !! 56.69867518484865 !! 57918.17
PP -31.11-- 56.119927781633834--- 56.92993611497764--57918.17--2024-07-10 00:00:00
PPBET -100.52-- 55.26859665846912--- 54.18329871539795--52.978554897942566 !! 57927.28--2024-07-10 00:00:00
PP 8.149999999999999-- 55.26859665846912--- 55.552964693694435--57918.17--2024-07-10 00:15:00
PP 57.519999999999996-- 54.18329871539795--- 53.79102293431387--57918.17--2024-07-10 00:30:00
PP 111.97-- 52.978554897942566--- 51.83823989099668--57918.17--2024-07-10 00:45:00
PP 111.97-- 52.978554897942566--- 51.83823989099668--57918.17 !! 57784.2--2024-07-10 00:45:00
2024

2024-08-20 23:45:00 !! 43.82143831413542 !! 44.91562459608944 !! 59310.36
PP 241.51-- 39.62199163042468--- 38.42522803748156--59310.36--2024-08-21 00:00:00
PP 202.43-- 40.50975647644851--- 39.81425153479099--59310.36--2024-08-21 00:15:00
PPBET 251.98000000000002-- 43.5639688656903--- 45.123723105577724--48.322548709636266 !! 59085.93--2024-08-21 00:15:00
PP 65.44-- 43.5639688656903--- 44.5373438965779--59310.36--2024-08-21 00:30:00
PPBET 122.46000000000001-- 45.123723105577724--- 48.322548709636266--47.180535351626155 !! 59222.92--2024-08-21 00:30:00
PP_old 101.215-- 45.123723105577724--- 44.5373438965779--59310.36 !! 59222.92--2024-08-21 00:30:00
2024-08-24 23:45:00 !! 49.76291134865922 !! 48.02295850259469 !! 64158.58
PP 43.370000000000005-- 47.984418387454966--- 45.42565624520281--64158.58--2024-08-25 00:00:00
PP 139.33-- 45.479119904998576--- 41.84765000411743--64158.58--2024-08-25 00:15:00
PP 157.21-- 45.01926144519184--- 41.19652787352099--64158.58--2024-08-25 00:30:00
PP 485.06-

In [381]:
n = 0
p = 0
tn = 0
tp = 0

for i in profit:
    if i<0.0:
        n = n+i
        tn = tn+1
    else:
        p = p+i
        tp = tp+1
print(sum(profit))
print(f"Total negative sm -->{n}")
print(f"Total negative -->{tn}")      
print(f"Total positive sm -->{p}")      
print(f"Total positive -->{tp}") 
print(f"Length {len(profit)}")

# print(time.time() - t1)
# -6, 6

-4944.585000000018
Total negative sm -->-41702.07
Total negative -->304
Total positive sm -->36757.48499999999
Total positive -->228
Length 532


In [373]:
profits.sort()

In [382]:
profits

[-24.78,
 -11.88,
 -42.620000000000005,
 80.02,
 34.72,
 -41.36,
 22.240000000000002,
 -30.34,
 -23.12,
 255.36,
 -65.34,
 9.780000000000001,
 92.78,
 -41.18,
 -77.66,
 25.659999999999997,
 25.380000000000003,
 242.60000000000002,
 -58.44,
 -28.560000000000002,
 -124.82,
 -24.0,
 -61.02,
 69.18,
 119.96000000000001,
 -24.52,
 -66.02000000000001,
 -14.66,
 -250.48,
 -35.62,
 -131.28,
 39.38,
 -168.42,
 118.91999999999999,
 -62.12,
 -39.9,
 -39.519999999999996,
 127.36000000000001,
 98.28,
 -88.68,
 -88.0,
 -2.039999999999999,
 20.36,
 -97.68,
 47.400000000000006,
 40.84,
 69.8,
 -39.42,
 -52.46,
 -35.94,
 -47.32,
 -85.62,
 -112.62,
 72.66,
 -144.45999999999998,
 -2.0,
 97.12,
 64.0,
 -44.260000000000005,
 -55.48,
 -2.8599999999999994,
 -95.92,
 -79.68,
 -27.66,
 -38.28,
 10.600000000000001,
 -837.2,
 -35.14,
 -3.780000000000001,
 -20.6,
 -46.9,
 8.54,
 -29.54,
 185.26,
 -7.4,
 -27.8,
 -10.52,
 -33.3,
 -238.24,
 47.44,
 -23.96,
 -29.08,
 -20.0,
 -46.5,
 -12.0,
 41.36,
 -25.62,
 -4834.1,
